# Used-Car Fair Price and Listing Risk Advisor - Colab Runner

This notebook runs the project pipeline in Google Colab while reusing the repository's Python scripts. It is a convenience runner, not a replacement for the script-based project structure.

**AI blocks used:** ML Numeric Data and NLP. No Computer Vision is used.

## Important Data Rule

This notebook does **not** download datasets automatically. It does **not** use the Kaggle API, Hugging Face datasets, scraping, or external data APIs.

You must manually download the Kaggle CSV files and upload them into Colab with these exact filenames:

- `data/raw/kaggle_craigslist_vehicles.csv`
- `data/raw/kaggle_carscom_used_cars.csv`

If the real files are missing, the project can create DEMO ONLY fallback data. That fallback is useful for testing code, but it is **not valid for final submission**.

## 1. Make the Repository Available in Colab

Recommended options:

1. Upload the whole `used-car-listing-advisor` folder to Google Drive, open this notebook from that folder, and run the cells.
2. Upload a zipped copy of the whole project to Colab, unzip it, and set `PROJECT_DIR` to the unzipped folder.
3. After publishing to GitHub, clone your repository into Colab, then upload the two Kaggle CSV files manually.

The notebook must be run from the project root or pointed to the project root. The project root is the folder that contains `app.py`, `requirements.txt`, and `src/`.

In [ ]:
from pathlib import Path
import os
import sys

# If you opened this notebook from the project root, leave this as Path.cwd().
# If needed, change it manually, for example:
# PROJECT_DIR = Path('/content/used-car-listing-advisor')
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / 'src').exists():
    candidates = list(Path('/content').glob('**/used-car-listing-advisor/src'))
    if candidates:
        PROJECT_DIR = candidates[0].parent

print('Project directory:', PROJECT_DIR)
assert (PROJECT_DIR / 'src').exists(), 'Could not find src/. Set PROJECT_DIR to the project root.'
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Working directory:', Path.cwd())

## 2. Install Requirements

This installs only the lightweight Python packages listed in `requirements.txt`.

In [ ]:
%pip install -q -r requirements.txt

## 3. Upload Local Kaggle CSV Files Manually

Run the cell below if the two raw CSV files are not already in `data/raw/`.

Upload files with the exact expected names:

- `kaggle_craigslist_vehicles.csv`
- `kaggle_carscom_used_cars.csv`

Do not upload the template CSVs as final data.

In [ ]:
from pathlib import Path

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

expected_files = [
    raw_dir / 'kaggle_craigslist_vehicles.csv',
    raw_dir / 'kaggle_carscom_used_cars.csv',
]

missing = [path.name for path in expected_files if not path.exists()]
print('Missing raw files:', missing if missing else 'None')

if missing:
    try:
        from google.colab import files
        print('Upload the missing CSV files now. They must already have the exact expected filenames.')
        uploaded = files.upload()
        for filename, content in uploaded.items():
            destination = raw_dir / filename
            destination.write_bytes(content)
            print('Saved:', destination)
    except ModuleNotFoundError:
        print('Not running in Colab. Manually copy CSV files into data/raw/.')

print('\nRaw directory contents:')
for path in sorted(raw_dir.glob('*')):
    print('-', path)

## 4. Load, Map, Validate, and Save Processed Data

This runs the same local data preparation script used by the repository.

In [ ]:
from src.data_loading import load_and_prepare_data

df = load_and_prepare_data()
display(df.head())
print('Processed shape:', df.shape)

## 5. Quick Data Checks

In [ ]:
import pandas as pd

processed = pd.read_csv('data/processed/project_listings.csv')
display(processed.head())
display(processed['source_name'].value_counts(dropna=False))
display(processed[['seller_price', 'year', 'mileage_km']].describe())
display(processed.isna().sum().sort_values(ascending=False).head(15))

## 6. Train Numeric Price Models

This compares Ridge Regression, Random Forest, and boosting-based regressors, then saves the best structured model and plots.

In [ ]:
from src.train_numeric import train_numeric_models

numeric_metrics = train_numeric_models()
numeric_metrics['best_model'], numeric_metrics['best_metrics']

## 7. Train NLP Risk Models

This compares negation-aware rule-based NLP with a TF-IDF + LogisticRegression classifier trained on weak labels.

In [ ]:
from src.train_nlp import train_nlp_models

nlp_metrics = train_nlp_models()
nlp_metrics['class_distribution'], nlp_metrics['tfidf_logistic_regression_metrics']['accuracy']

## 8. Train Integrated Structured + NLP Model

This compares a structured-only model with a model that also uses NLP-derived features.

In [ ]:
from src.train_integrated import train_integrated_models

integrated_metrics = train_integrated_models()
integrated_metrics['best_integrated_model'], integrated_metrics['interpretation']

## 9. Evaluate Saved Models and Reports

In [ ]:
from src.evaluate import evaluate_saved_models

evaluate_saved_models()

## 10. Inspect Generated Reports and Figures

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

for report in ['metrics_numeric.json', 'metrics_nlp.json', 'metrics_integrated.json']:
    path = Path('reports') / report
    print('\n---', path, '---')
    with path.open('r', encoding='utf-8') as file:
        data = json.load(file)
    print(json.dumps(data, indent=2)[:2500])

figure_paths = sorted(Path('reports/figures').glob('*.png'))
print('\nFigures:', [str(path) for path in figure_paths])
for path in figure_paths[:4]:
    print(path)
    display(Image(filename=str(path)))

## 11. Run One Inference Example

In [ ]:
from src.inference import predict_listing
import json

example_result = predict_listing(
    brand='BMW',
    model='328i',
    year=2013,
    mileage_km=171000,
    fuel_type='Petrol',
    transmission='Automatic',
    body_type='Sedan',
    engine_size_l=2.0,
    condition='fair',
    title_status='rebuilt',
    drive='rwd',
    paint_color='gray',
    location_country='USA',
    location_region='fl',
    currency='USD',
    seller_price=9600,
    seller_description='BMW 328i sold as is, oil leak, check engine light, needs repair, mechanic special.'
)

print(json.dumps(example_result, indent=2))

## 12. Optional: Launch the Gradio App in Colab

Run this after training. In Colab, `share=True` is usually the easiest way to open the Gradio interface. The app uses local saved models and does not download datasets.

In [ ]:
# Optional UI launch. Stop this cell when you are done using the app.
import app

app.demo.launch(share=True, debug=False)

## 13. Final Submission Reminders

- Do not submit metrics from DEMO ONLY fallback data.
- Rerun the full notebook or script workflow after uploading the real Kaggle CSV files.
- Copy final metrics from `reports/*.json` into `documentation.md`.
- Add the GitHub repository URL, deployment URL, student name, and screenshots.
- Add required GitHub users: `jasminh` and `bkuehnis`.
- Verify that no forbidden datasets or Computer Vision components are included.